# Spinal Disorder Classification — Stage 3C: CLAHE Hyperparameter Tuning

**Objective:** Test whether changing CLAHE contrast enhancement can improve the validation performance of our **reference CNN**. This is a follow-on experiment to Stages 1–3B, using the **exact same frozen v1 train/validation/test split**.

**What stays fixed:**
- Three-class label mapping, 235 training images, 52 validation images, **51 untouched test images**.
- Reference architecture from **E0 / E0-C1**: three Conv2D→MaxPool blocks, Flatten, Dense(128), Dense(3).
- Image size **128×128 grayscale**, normalized to `[0, 1]`.
- Adam LR `3e-4`, batch size 16, up to 40 epochs, class weights, early stopping, checkpointing, random seed **42**.

**What changes:** Only OpenCV CLAHE `clipLimit` and `tileGridSize`.

| | Tile 4×4 | Tile 8×8 | Tile 16×16 |
|---|---|---|---|
| Clip 1.0 | C01 | C02 | C03 |
| Clip 2.0 | C04 | **C05 = prior E0-C1** | C06 |
| Clip 3.0 | C07 | C08 | C09 |

**Run plan:** Start with **C02 and C08**, keeping 8×8 tiles while changing clip limit. Then compare grid size using C04/C06. If needed, expand to C01 and C07. At 128×128 image resolution, the 16×16 tile variants C03/C06/C09 can be effectively identical due to CLAHE clipping quantization; we will use C06 as their representative and avoid redundant runs.

**Very important:** We evaluate only the **validation set** when choosing hyperparameters. Reusing a small 52-image validation set across many configurations risks overfitting to it. Scores here cannot establish clinical usefulness or patient-independent accuracy, because patient IDs are unavailable. The reserved test set is only for one final evaluation after model selection.

**Privacy:** Dataset and private manifests remain in Google Drive; do not put original images, private filenames or private checkpoint artifacts in a public GitHub repository. This is a research demonstration, not a diagnostic medical device.


## 1. Mount Google Drive

Use the same Google account that contains `224/` and the frozen Stage 2 split.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Dependencies and project paths

Each new trial saves into a **new** directory under `experiments/stage3/clahe_grid_v1/`. Previous experiments remain untouched.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = Path('/content/drive/MyDrive/224')
SPLIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1')
EXP_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/experiments/stage3')
EXP_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TO_NAME = {0: 'Normal', 1: 'Scoliosis', 2: 'Spondylolisthesis'}
CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
GRID_DIR = EXP_DIR / 'clahe_grid_v1'
print('Existing experiment folder:', EXP_DIR)
print('New CLAHE grid folder:', GRID_DIR)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print('CPU-only training detected. GPU can speed up the grid search.')
if not tf.config.list_physical_devices('GPU'):
    print('NOTE: No GPU detected. Training can run on CPU but will be slower.')

## 3. Load and verify frozen split v1

Do not regenerate the split. This checks the saved dataset checksum manifest and uses only train/validation records.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
config_path = SPLIT_DIR / 'split_config_PRIVATE.json'

assert manifest_path.is_file() and config_path.is_file(), (
    'Frozen Stage 2 split not found. Run Stage 2 first; do not generate a new split.'
)
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns), f'Missing manifest columns: {required - set(manifest.columns)}'
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['seed'] == SEED
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1, 'Grouped images cross splits.'
assert all(int(row.label) == CLASS_TO_LABEL[row.class_name] for row in manifest.itertuples(index=False))

rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
        .sort_values('relative_path')
        .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Frozen dataset fingerprint mismatch.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235 and counts['validation'].sum() == 52 and counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
# Intentional: no test_df or test image loader in this notebook.
assert set(train_df['relative_path']).isdisjoint(val_df['relative_path'])
print('Freeze verified. Development images:', len(train_df) + len(val_df))
print('Test set RESERVEd for final evaluation:', int(counts['test'].sum()))

## 4. Image preprocessing: adjustable CLAHE

This matches the previous E0-C1 pipeline except that CLAHE clip limit and tile size are explicit arguments. Every image is treated identically regardless of diagnosis. No preprocessing parameters are learned from the validation or test images.

In [ ]:
from PIL import Image, ImageOps

# OpenCV CLAHE is a per-image deterministic transform; it does not fit dataset statistics.
def preprocess_clahe(path, clip_limit, tile_grid, size=128):
    if not (0.0 < clip_limit <= 10.0):
        raise ValueError('clip_limit must be positive and at most 10 for this notebook')
    if not isinstance(tile_grid, tuple) or len(tile_grid) != 2:
        raise ValueError('tile_grid must be a (rows, cols) tuple')

    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(
            original, (size, size), method=Image.Resampling.LANCZOS,
            color=(0, 0, 0),
        )
        rgb = np.asarray(resized, dtype=np.uint8)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    enhanced = cv2.createCLAHE(
        clipLimit=float(clip_limit), tileGridSize=tuple(tile_grid)
    ).apply(gray)
    return enhanced[..., np.newaxis].astype(np.float32) / 255.0


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_tuned_partition(rows, clip_limit, tile_grid):
    X, y = [], []
    for item in rows.itertuples(index=False):
        path = DATA_ROOT / item.relative_path
        if not path.is_file() or file_sha256(path) != item.file_sha256:
            raise RuntimeError('An image changed since the frozen split; stop and review.')
        X.append(preprocess_clahe(path, clip_limit, tile_grid))
        y.append(int(item.label))
    return np.stack(X), np.asarray(y, dtype=np.int64)

# Sanity check on training image only.
example_path = DATA_ROOT / train_df.iloc[0]['relative_path']
for clip in (1.0, 2.0, 3.0):
    arr = preprocess_clahe(example_path, clip, (8, 8))
    assert arr.shape == (128, 128, 1) and arr.dtype == np.float32
    assert np.isfinite(arr).all() and 0 <= arr.min() <= arr.max() <= 1
print('CLAHE image preprocessing sanity checks passed.')


## 5. Reference CNN: identical architecture to E0-C1

This is intentionally **not** the BatchNormalization model, which collapsed during inference. We retain the reference CNN to isolate CLAHE parameters.

In [ ]:
from tensorflow.keras import layers

def make_reference_cnn(shape):
    return tf.keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ], name='reference_cnn')

def make_model(shape):
    model = make_reference_cnn(shape)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model

# Inspect architecture using the correct one-channel input.
preview = make_model((128, 128, 1))
print('Reference CNN parameters (grayscale):', preview.count_params())
del preview
tf.keras.backend.clear_session()


## 6. Register the CLAHE search grid and verify the original baseline

**C05 = existing `E0-C1`.** We reuse its validation metrics and original saved checkpoint without retraining. If the baseline file or configuration is missing or different, this notebook stops rather than pretending it is comparable.

In [ ]:
from itertools import product

GRID = {}
for index, (clip_limit, tiles) in enumerate(
    product((1.0, 2.0, 3.0), (4, 8, 16)), start=1
):
    trial_id = f'C{index:02d}'
    GRID[trial_id] = {
        'clip_limit': float(clip_limit),
        'tile_grid': (int(tiles), int(tiles)),
    }
assert GRID['C05'] == {'clip_limit': 2.0, 'tile_grid': (8, 8)}

BASELINE_ID = 'E0-C1'
BASELINE_DIR = EXP_DIR / BASELINE_ID
baseline_config_path = BASELINE_DIR / 'configuration.json'
baseline_metrics_path = BASELINE_DIR / 'validation_metrics.json'
assert baseline_config_path.is_file() and baseline_metrics_path.is_file(), (
    'E0-C1 baseline results not found. Run Stage 3B CLAHE baseline first.'
)

baseline_config = json.loads(baseline_config_path.read_text(encoding='utf-8'))
baseline_metrics = json.loads(baseline_metrics_path.read_text(encoding='utf-8'))
expected_settings = {
    'experiment': BASELINE_ID,
    'architecture': 'reference',
    'mode': 'clahe',
    'size': 128,
    'seed': SEED,
    'batch_size': 16,
    'epochs_max': 40,
    'dataset_fingerprint': fingerprint,
    'split_version': 'v1',
    'train_images': 235,
    'validation_images': 52,
    'normalization': 'fixed pixel division by 255',
    'clahe_settings': {'clipLimit': 2.0, 'tileGridSize': [8, 8]},
    'augmentation': 'none (matches E0)',
    'optimizer': 'Adam',
    'initial_learning_rate': 3e-4,
    'checkpoint_monitor': 'val_loss',
}
for key, expected_value in expected_settings.items():
    if baseline_config.get(key) != expected_value:
        raise RuntimeError(f'E0-C1 baseline configuration mismatch: {key}')
assert baseline_metrics.get('test_evaluated') is False
assert (BASELINE_DIR / 'best_model.keras').is_file(), 'Baseline checkpoint missing.'
print('Verified original CLAHE baseline E0-C1 = C05.')

# Begin with only two new trials: test different clip limits at the same tile size.
RUN_IDS = ['C02', 'C08']

# Future batches: ['C04', 'C06'], then optionally ['C01', 'C07'].
# At 128 pixels, clip 1/2/3 with 16x16 tiles may be effectively equivalent.
# C06 represents those variants; avoid wasting training time on C03 and C09.
# All nine configurations are defined above. Never include C05 here.
EPOCHS = 40
BATCH_SIZE = 16
assert len(set(RUN_IDS)) == len(RUN_IDS)
assert all(trial_id in GRID and trial_id not in {'C05', 'C03', 'C09'} for trial_id in RUN_IDS), (
    'C05 is already completed; C03/C09 duplicate the 16x16 clipping settings at 128px.'
)
print('Selected new trials:', RUN_IDS)

display(pd.DataFrame([
    {'trial': tid, **{'clip_limit': cfg['clip_limit'], 'tile_grid': str(cfg['tile_grid'])},
     'source': 'existing E0-C1' if tid == 'C05' else 'new trial'}
    for tid, cfg in GRID.items()
]))


## 7. Training function: append-only saved experiments

Each run saves `best_model.keras`, `history.csv`, `validation_metrics.json`, configuration and learning curves. A completed trial is **read and skipped** on reruns. Incomplete folders are not overwritten. Results depend on the saved validation-loss checkpoint, matching the earlier protocol.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight


def train_trial(trial_id):
    if trial_id == 'C05':
        print('C05 corresponds to existing E0-C1; no training required.')
        return dict(baseline_metrics, trial='C05', clip_limit=2.0, tile_grid='8x8')
    if trial_id not in GRID:
        raise ValueError(f'Unknown trial: {trial_id}')

    setting = GRID[trial_id]
    clip_limit, tile_grid = setting['clip_limit'], setting['tile_grid']
    folder = GRID_DIR / trial_id
    config_path = folder / 'configuration.json'
    result_path = folder / 'validation_metrics.json'
    model_path = folder / 'best_model.keras'

    # Reusing a finished experiment NEVER overwrites its saved data.
    if folder.exists():
        if not all(p.is_file() for p in (config_path, result_path, model_path)):
            raise RuntimeError(
                f'Partial saved trial exists: {folder}. Do not delete or overwrite automatically.'
            )
        saved = json.loads(config_path.read_text(encoding='utf-8'))
        expected = {
            'trial': trial_id, 'clip_limit': clip_limit,
            'tile_grid': list(tile_grid), 'dataset_fingerprint': fingerprint,
            'split_version': 'v1', 'seed': SEED,
            'epochs_max': EPOCHS, 'batch_size': BATCH_SIZE,
        }
        for key, value in expected.items():
            if saved.get(key) != value:
                raise RuntimeError(f'{trial_id}: incompatible saved setting {key}; stop.')
        metrics = json.loads(result_path.read_text(encoding='utf-8'))
        assert metrics.get('test_evaluated') is False
        print(f'{trial_id}: reusing completed saved results (no retraining).')
        return metrics

    # Training and validation ONLY, from the frozen v1 manifest.
    X_train, y_train = load_tuned_partition(train_df, clip_limit, tile_grid)
    X_val, y_val = load_tuned_partition(val_df, clip_limit, tile_grid)
    assert X_train.shape == (235, 128, 128, 1)
    assert X_val.shape == (52, 128, 128, 1)
    assert len(np.unique(y_train)) == 3

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = make_model(X_train.shape[1:])
    weights = compute_class_weight('balanced', classes=np.arange(3), y=y_train)
    class_weights = {int(label): float(value) for label, value in enumerate(weights)}

    settings = {
        'trial': trial_id,
        'architecture': 'reference',
        'preprocessing': 'grayscale_clahe',
        'clip_limit': clip_limit,
        'tile_grid': list(tile_grid),
        'resize': [128, 128],
        'normalization': 'divide by 255.0 after CLAHE',
        'seed': SEED,
        'epochs_max': EPOCHS,
        'batch_size': BATCH_SIZE,
        'optimizer': 'Adam',
        'learning_rate': 3e-4,
        'checkpoint_monitor': 'val_loss',
        'class_weights': class_weights,
        'train_images': int(len(y_train)),
        'validation_images': int(len(y_val)),
        'patient_independent': False,
        'test_evaluated': False,
        'dataset_fingerprint': fingerprint,
        'split_version': 'v1',
        'parameters': int(model.count_params()),
        'tensorflow_version': str(tf.__version__),
        'opencv_version': str(cv2.__version__),
    }

    folder.mkdir(parents=True, exist_ok=False)
    config_path.write_text(json.dumps(settings, indent=2), encoding='utf-8')

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(model_path), monitor='val_loss', save_best_only=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=9, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(folder / 'epoch_log.csv')),
    ]

    print(f'\n=== {trial_id}: CLAHE clipLimit={clip_limit}, tiles={tile_grid} ===')
    start = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    best = tf.keras.models.load_model(str(model_path))
    scores = best.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    pred = np.argmax(scores, axis=1)
    cm = confusion_matrix(y_val, pred, labels=[0, 1, 2])
    report = classification_report(
        y_val, pred, labels=[0, 1, 2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True, zero_division=0,
    )

    result = {
        'trial': trial_id,
        'experiment': trial_id,
        'clip_limit': clip_limit,
        'tile_grid': f'{tile_grid[0]}x{tile_grid[1]}',
        'val_accuracy': float(accuracy_score(y_val, pred)),
        'val_macro_f1': float(f1_score(
            y_val, pred, labels=[0, 1, 2], average='macro', zero_division=0
        )),
        'val_balanced_accuracy': float(balanced_accuracy_score(y_val, pred)),
        'best_val_loss': float(min(history.history['val_loss'])),
        'best_epoch': int(np.argmin(history.history['val_loss']) + 1),
        'epochs_completed': int(len(history.history['loss'])),
        'runtime_seconds': float(time.perf_counter() - start),
        'per_class_recall': {
            name: float(report[name]['recall']) for name in LABEL_TO_NAME.values()
        },
        'test_evaluated': False,
    }

    pd.DataFrame(history.history).to_csv(folder / 'history.csv', index=False)
    np.save(folder / 'confusion_matrix_validation.npy', cm)
    (folder / 'classification_report.json').write_text(
        json.dumps(report, indent=2), encoding='utf-8'
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for metric, ax in zip(('loss', 'accuracy'), axes):
        ax.plot(history.history[metric], label='train')
        ax.plot(history.history['val_' + metric], label='validation')
        ax.set_xlabel('Epoch')
        ax.set_title(f'{trial_id}: {metric}')
        ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'learning_curves.png', dpi=150)
    plt.show()
    plt.close(fig)

    # Written LAST: marker of a completed run (all other artifacts exist).
    result_path.write_text(json.dumps(result, indent=2), encoding='utf-8')

    print('Validation confusion matrix: rows are actual labels')
    display(pd.DataFrame(cm, index=list(LABEL_TO_NAME.values()),
                         columns=list(LABEL_TO_NAME.values())))
    print('Validation-only results:')
    print(json.dumps(result, indent=2))
    return result


## 8. Run selected trials

Default is **two** new trials, `C02` and `C08` (clip-limit comparisons at fixed 8×8 tiles). Re-run this cell safely: completed trials will be verified and skipped.

In [ ]:
results = []
for trial_id in RUN_IDS:
    results.append(train_trial(trial_id))

display(pd.DataFrame(results).drop(columns=['per_class_recall'], errors='ignore').round(4))


## 9. Validation-only leaderboard

This reads the saved C05 baseline from E0-C1 and every completed new CLAHE run. It ranks **macro F1 first**, then balanced accuracy, then loss; this is *not* a test-set leaderboard. Rerunning it never retrains or changes checkpoints.

In [ ]:
completed = []

# Import the previous baseline with the parameters it actually used.
completed.append({
    **baseline_metrics,
    'trial': 'C05',
    'experiment': 'E0-C1 (baseline)',
    'clip_limit': 2.0,
    'tile_grid': '8x8',
})

for trial_id in GRID:
    if trial_id == 'C05':
        continue
    folder = GRID_DIR / trial_id
    path = folder / 'validation_metrics.json'
    if not path.exists():
        continue
    saved_config = json.loads((folder / 'configuration.json').read_text('utf-8'))
    if saved_config.get('dataset_fingerprint') != fingerprint:
        raise RuntimeError(f'{trial_id} used a different frozen dataset!')
    metric = json.loads(path.read_text('utf-8'))
    assert metric['test_evaluated'] is False
    completed.append(metric)

leaderboard = pd.DataFrame(completed).sort_values(
    ['val_macro_f1', 'val_balanced_accuracy', 'best_val_loss'],
    ascending=[False, False, True],
)
show_cols = [
    'trial', 'clip_limit', 'tile_grid', 'val_accuracy', 'val_macro_f1',
    'val_balanced_accuracy', 'best_val_loss', 'best_epoch', 'epochs_completed',
]
print('Validation results; all 51 held-out test images remain untouched.')
display(leaderboard[show_cols].round(4).reset_index(drop=True))
print('Current CLAHE-only validation leader:', leaderboard.iloc[0]['trial'])


## Next: decisions before the final test

- **Batch 1 (default):** C02 (clip 1.0, tiles 8×8) and C08 (clip 3.0, tiles 8×8).
- **Batch 2:** Change `RUN_IDS` in Section 6 to `['C04', 'C06']` to compare tile size while keeping clip limit 2.0.
- **Batch 3 (optional):** `['C01','C07']`. Because of CLAHE's clip-limit quantization with 128×128 inputs and 16×16 tile grids, C03/C06/C09 can generate indistinguishable preprocessing; C06 is the representative. Avoid spending unlimited experiments on this small validation set.
- If a hyperparameter combination looks better, examine the **per-class recall, macro F1, validation loss, confusion matrix**, and stability across repeat seeds or development-only cross-validation before selecting it.
- Once the model and all preprocessing choices are finalized, evaluate the held-out **51-image test set once**. Patient independence is still unverified.
- For GitHub, commit a **cleared-output notebook** and aggregate validation summaries only; do not publish source images or private patient-identifying filenames.
